# Exercise 2 - Population Dynamics in a Lake

This notebook solves all four parts of the exercise using Python.

The goal is not to memorize difficult formulas. We will use a simple idea:

> A matrix is a set of rules that tells us how today's population becomes next year's population.

We track three populations in this order:

1. algae
2. small fish
3. large fish

Run the notebook from top to bottom. Each code cell is followed by an explanation of what the result means.

## 0. Import the tools

- `numpy` helps us work with vectors and matrices.
- `matplotlib` helps us draw population graphs.

Both libraries are already available in Google Colab.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=6, suppress=True)

## 1. Define the lake model

The matrix from the exercise is:

$$
M =
\begin{bmatrix}
0.80 & 0.15 & 0.05 \\
0.15 & 0.70 & 0.15 \\
0.05 & 0.15 & 0.80
\end{bmatrix}
$$

The starting vector is:

$$
x_1 =
\begin{bmatrix}
10 \\ 20 \\ 4
\end{bmatrix}
$$

For example, `x1[0]` is the algae population and `x1[2]` is the large-fish population.
The rule `x_next = M @ x_current` means "multiply the matrix by the current population vector."

In [ ]:
species = ["Algae", "Small fish", "Large fish"]

M = np.array([
    [0.80, 0.15, 0.05],
    [0.15, 0.70, 0.15],
    [0.05, 0.15, 0.80],
])

x1 = np.array([10.0, 20.0, 4.0])

print("Transition matrix M:")
print(M)
print("\nYear 1 population x1:")
for name, population in zip(species, x1):
    print(f"{name}: {population}")

# Part 1 - Computing Future Populations

## 1.1 Compute $x_2$

We apply the transition rule one time:

$$x_2 = Mx_1$$

The `@` symbol is Python's matrix-multiplication operator.

In [ ]:
x2 = M @ x1

print("x2 =", x2)
for name, population in zip(species, x2):
    print(f"{name}: {population:.3f}")

So the Year 2 vector is:

$$
x_2 =
\begin{bmatrix}
11.2 \\ 16.1 \\ 6.7
\end{bmatrix}
$$

These values do not need to be whole numbers. This is a mathematical population model, so decimal values are allowed.

## 1.2 Compute $x_3$

Year 3 comes from Year 2:

$$x_3 = Mx_2$$

This is the same as applying the matrix twice to the starting vector:

$$x_3 = M^2x_1$$

In [ ]:
x3 = M @ x2

print("x3 =", x3)
for name, population in zip(species, x3):
    print(f"{name}: {population:.3f}")

Therefore:

$$
x_3 =
\begin{bmatrix}
11.71 \\ 13.955 \\ 8.335
\end{bmatrix}
$$

## 1.3 Verify that the total population stays constant

We add the three entries in each vector. The starting total is:

$$10 + 20 + 4 = 34$$

If the model conserves population, every future total should also be `34`.

In [ ]:
for year, population_vector in [(1, x1), (2, x2), (3, x3)]:
    print(f"Year {year} total: {population_vector.sum():.6f}")

print("\nColumn sums of M:", M.sum(axis=0))

The total remains `34`.

Why? Every column of `M` sums to `1`. This means all of the population represented by each current group is redistributed among the three groups; none is added or removed by the model.

# Part 2 - Eigenvalues and Eigenvectors

## A friendly meaning

An **eigenvector** is a special direction that a matrix does not turn into a different direction.
The matrix may stretch or shrink it, but the direction stays the same.

The **eigenvalue** tells us the stretching factor:

$$Mv = \lambda v$$

- If $\lambda = 1$, that direction stays the same size.
- If $|\lambda| < 1$, that part becomes smaller each time we apply the matrix.

Because `M` is symmetric, `np.linalg.eigh` gives especially stable real-valued results.

In [ ]:
eigenvalues, eigenvectors = np.linalg.eigh(M)

# Show the largest eigenvalue first.
order = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[order]
eigenvectors = eigenvectors[:, order]

print("Eigenvalues:")
print(eigenvalues)

print("\nNormalized eigenvectors (stored as columns):")
print(eigenvectors)

The eigenvalues are:

$$1,\quad 0.75,\quad 0.55$$

Python returns eigenvectors with length `1`, and their signs may be reversed. That is normal: if `v` is an eigenvector, then `-v` is also an eigenvector.

For easier reading, we can use these scaled versions:

| Eigenvalue | Easy-to-read eigenvector |
|---:|---|
| `1.00` | `[1, 1, 1]` |
| `0.75` | `[1, 0, -1]` |
| `0.55` | `[1, -2, 1]` |

In [ ]:
easy_eigenvectors = [
    (1.00, np.array([1.0, 1.0, 1.0])),
    (0.75, np.array([1.0, 0.0, -1.0])),
    (0.55, np.array([1.0, -2.0, 1.0])),
]

for eigenvalue, vector in easy_eigenvectors:
    left_side = M @ vector
    right_side = eigenvalue * vector
    print(f"lambda = {eigenvalue:.2f}, vector = {vector}")
    print("M @ vector       =", left_side)
    print("lambda * vector  =", right_side)
    print("Match?", np.allclose(left_side, right_side))
    print()

## 2.1 Verify the eigenvalue `1` and vector `[1, 1, 1]`

The first row gives:

$$0.80(1) + 0.15(1) + 0.05(1) = 1$$

The other rows also give `1`, so:

$$
M
\begin{bmatrix}
1 \\ 1 \\ 1
\end{bmatrix}
=
\begin{bmatrix}
1 \\ 1 \\ 1
\end{bmatrix}
=
1
\begin{bmatrix}
1 \\ 1 \\ 1
\end{bmatrix}
$$

Therefore `[1, 1, 1]` is an eigenvector and its eigenvalue is `1`.

In [ ]:
ones_vector = np.ones(3)

print("M @ [1, 1, 1] =", M @ ones_vector)
print("Equal to [1, 1, 1]?", np.allclose(M @ ones_vector, ones_vector))

# Part 3 - Long-Term Behavior

The exercise defines Year 1 as the starting vector. To reach Year 30, we apply the transition 29 times:

$$x_{30} = M^{29}x_1$$

`np.linalg.matrix_power(M, 29)` calculates $M^{29}$ for us.

In [ ]:
x30 = np.linalg.matrix_power(M, 29) @ x1
equal_share = np.full(3, x1.sum() / 3)

print("x30 =", x30)
print("Equal one-third share =", equal_share)
print("Difference =", x30 - equal_share)

The result is approximately:

$$
x_{30} =
\begin{bmatrix}
11.3340 \\ 11.3333 \\ 11.3326
\end{bmatrix}
$$

The total is `34`, so an equal distribution would give each group:

$$\frac{34}{3} \approx 11.3333$$

The Year 30 vector is almost exactly a multiple of `[1, 1, 1]`.

## Why does this happen?

The starting vector can be thought of as a mixture of the three eigenvector directions.

- The part connected to eigenvalue `1` remains.
- The part connected to `0.75` is multiplied by `0.75` again and again.
- The part connected to `0.55` is multiplied by `0.55` again and again.

Because `0.75^n` and `0.55^n` approach zero, only the equal-distribution direction remains in the long run.

## 3.1 Plot the first 30 years

The graph makes the convergence easier to see. Each line moves toward the same value.

In [ ]:
history = [x1]

for _ in range(29):
    history.append(M @ history[-1])

history = np.array(history)
years = np.arange(1, 31)

for index, name in enumerate(species):
    plt.plot(years, history[:, index], marker="o", markersize=3, label=name)

plt.axhline(x1.sum() / 3, color="black", linestyle="--", label="Equal share")
plt.xlabel("Year")
plt.ylabel("Population")
plt.title("Population convergence over 30 years")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# Part 4 - Different Initial Conditions

Now we repeat the Year 30 calculation for three very different starting vectors:

$$
\begin{bmatrix}30\\5\\1\end{bmatrix},
\quad
\begin{bmatrix}2\\50\\10\end{bmatrix},
\quad
\begin{bmatrix}100\\0\\0\end{bmatrix}
$$

For a fair comparison, we will show both:

1. the actual Year 30 population;
2. the **proportion** in each group, found by dividing by the total.

In [ ]:
initial_conditions = {
    "Start A: [30, 5, 1]": np.array([30.0, 5.0, 1.0]),
    "Start B: [2, 50, 10]": np.array([2.0, 50.0, 10.0]),
    "Start C: [100, 0, 0]": np.array([100.0, 0.0, 0.0]),
}

results = {}

for label, initial_vector in initial_conditions.items():
    final_vector = np.linalg.matrix_power(M, 29) @ initial_vector
    final_proportions = final_vector / final_vector.sum()
    results[label] = final_vector

    print(label)
    print("  Initial total:", initial_vector.sum())
    print("  x30:", final_vector)
    print("  x30 proportions:", final_proportions)
    print()

## 4.1 Compare the results

The actual Year 30 values are different because the starting totals are different:

- `[30, 5, 1]` has total `36`, so each long-term group approaches `12`.
- `[2, 50, 10]` has total `62`, so each group approaches `62 / 3 = 20.6667`.
- `[100, 0, 0]` has total `100`, so each group approaches `100 / 3 = 33.3333`.

But the proportions are almost the same:

$$
\left[\frac13,\frac13,\frac13\right]
$$

So the model does **not** force every experiment to the same total number. It forces the long-term **distribution** toward one-third algae, one-third small fish, and one-third large fish.

In [ ]:
labels = list(results.keys())
proportions = np.array([results[label] / results[label].sum() for label in labels])
x_positions = np.arange(len(species))
bar_width = 0.25

for index, label in enumerate(labels):
    plt.bar(
        x_positions + (index - 1) * bar_width,
        proportions[index],
        width=bar_width,
        label=label,
    )

plt.axhline(1 / 3, color="black", linestyle="--", label="One third")
plt.xticks(x_positions, species)
plt.ylabel("Share of total population")
plt.title("Year 30 distributions from different starting populations")
plt.ylim(0, 0.4)
plt.legend(fontsize=8)
plt.grid(axis="y", alpha=0.3)
plt.show()

# Final Answers

## Part 1

- $x_2 = [11.2,\ 16.1,\ 6.7]^T$
- $x_3 = [11.71,\ 13.955,\ 8.335]^T$
- The total remains `34`.

## Part 2

- Eigenvalues: `1`, `0.75`, and `0.55`.
- Corresponding easy-to-read eigenvectors: `[1, 1, 1]`, `[1, 0, -1]`, and `[1, -2, 1]`.
- `[1, 1, 1]` has eigenvalue `1`.

## Part 3

- $x_{30} \approx [11.3340,\ 11.3333,\ 11.3326]^T$.
- It is very close to $\frac{34}{3}[1,1,1]^T$.

## Part 4

- Different starting totals produce different final amounts.
- All three examples approach the same equal one-third distribution.
- The reason is that eigenvalue `1` preserves the `[1,1,1]` direction, while the effects of eigenvalues `0.75` and `0.55` shrink over time.